**Cell 1**

# 02 — Combine Per-Class Sources into the Unified Split Dataset

The per-class downloads from notebook 01 are each single-class Roboflow exports shaped like:

```
data/sources/<class>/
├── train/{images,labels}
├── valid/{images,labels}
├── test/{images,labels}
└── data.yaml
```

This notebook merges all four classes into one **split-preserving** dataset and remaps every `class_id` from `0` to the global index (`projector=0`, `whiteboard=1`, `fire_extinguisher=2`, `door_sign=3`):

```
data/dataset/
├── images/{train,val,test}/
├── labels/{train,val,test}/
└── data.yaml
```

Roboflow uses `valid/` — we rename it to `val/` here. If a source is flat (no split folders), all of its files go to `train/` and you should split it with notebook 03's helper (section 3.x).

In [ ]:
# Cell 2 — install dependencies for this notebook
%pip install -q pandas pillow matplotlib pyyaml

In [ ]:
# Cell 3 — setup
from pathlib import Path
import pandas as pd
import yaml
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

random.seed(42)
CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
GLOBAL_ID = {c: i for i, c in enumerate(CLASSES)}
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
# Roboflow → our split name
SPLIT_ALIAS = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

SRC = Path('../data/sources')
DST = Path('../data/dataset')
for s in ['train', 'val', 'test']:
    (DST / 'images' / s).mkdir(parents=True, exist_ok=True)
    (DST / 'labels' / s).mkdir(parents=True, exist_ok=True)

**Cell 4**

## 2.1 Discover (image, label, split) triples inside each class folder
For a Roboflow export the split is taken from the ancestor directory (`train` / `valid` / `test`). For a flat Kaggle layout with no such ancestor we fall back to `train`.

In [ ]:
# Cell 5 — walk each class and infer the split
def infer_split(img_path: Path) -> str:
    for part in img_path.parts[::-1]:
        if part in SPLIT_ALIAS:
            return SPLIT_ALIAS[part]
    return 'train'  # flat layout fallback

def find_label(img_path: Path) -> Path | None:
    for cand in [img_path.parent.parent / 'labels' / (img_path.stem + '.txt'),
                 img_path.with_suffix('.txt')]:
        if cand.exists():
            return cand
    return None

triples = []
for c in CLASSES:
    folder = SRC / c
    if not folder.exists():
        print(f'[missing] {folder}'); continue
    for img in folder.rglob('*'):
        if img.suffix.lower() not in IMG_EXTS: continue
        lbl = find_label(img)
        if lbl is None: continue
        triples.append({'class': c, 'img': img, 'lbl': lbl, 'split': infer_split(img)})

df = pd.DataFrame(triples)
df.groupby(['class', 'split']).size().unstack(fill_value=0)

**Cell 6**

## 2.2 Merge — remap `class_id`, rename files, copy into the right split

In [ ]:
# Cell 7 — write images + remapped labels into data/dataset/{split}/
counter = {(c, s): 0 for c in CLASSES for s in ['train', 'val', 'test']}
manifest = []
for row in df.itertuples():
    counter[(row._1, row.split)] += 1
    idx = counter[(row._1, row.split)]
    stem = f'{row._1}_{row.split}_{idx:04d}'

    dst_img = DST / 'images' / row.split / f'{stem}.jpg'
    dst_lbl = DST / 'labels' / row.split / f'{stem}.txt'

    with Image.open(row.img) as im:
        im.convert('RGB').save(dst_img, 'JPEG', quality=92)

    gid = GLOBAL_ID[row._1]
    remapped = []
    for line in row.lbl.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) != 5: continue
        _, xc, yc, w, h = parts
        remapped.append(f'{gid} {xc} {yc} {w} {h}')
    dst_lbl.write_text('\n'.join(remapped) + ('\n' if remapped else ''))

    manifest.append({'filename': dst_img.name, 'class': row._1,
                     'global_id': gid, 'split': row.split, 'boxes': len(remapped)})

mf = pd.DataFrame(manifest)
mf.to_csv(DST / 'manifest.csv', index=False)
mf.groupby(['split', 'class']).agg(images=('filename','count'), boxes=('boxes','sum'))

**Cell 8**

## 2.3 Structural validation
Every image has a matching label; every row has 5 numeric fields; class_id in `[0,3]`; coords in `[0,1]`.

In [ ]:
# Cell 9 — validate labels across all splits
issues = []
for split in ['train', 'val', 'test']:
    img_dir = DST / 'images' / split; lbl_dir = DST / 'labels' / split
    for img in img_dir.glob('*.jpg'):
        lbl = lbl_dir / (img.stem + '.txt')
        if not lbl.exists():
            issues.append((split, img.name, 'missing_label')); continue
        for ln, line in enumerate(lbl.read_text().strip().splitlines(), 1):
            parts = line.split()
            if len(parts) != 5:
                issues.append((split, img.name, f'bad_row_{ln}')); continue
            cid, *coords = parts
            if int(cid) not in range(len(CLASSES)):
                issues.append((split, img.name, f'bad_class_{cid}'))
            if any(not (0 <= float(x) <= 1) for x in coords):
                issues.append((split, img.name, f'coords_out_of_range_row_{ln}'))

pd.DataFrame(issues, columns=['split', 'file', 'issue']).head(20)

**Cell 10**

## 2.4 Visual spot-check — confirm the class remap is correct
If a `whiteboard_*` image is drawn with a `projector` label, the remap is wrong for the whiteboard source.

In [ ]:
# Cell 11 — overlay 3 train images per class
def draw(ax, img_path, lbl_path):
    im = Image.open(img_path); W, H = im.size
    ax.imshow(im); ax.set_title(img_path.name, fontsize=8); ax.axis('off')
    for line in lbl_path.read_text().strip().splitlines():
        cid, xc, yc, w, h = map(float, line.split())
        x = (xc - w/2) * W; y = (yc - h/2) * H
        ax.add_patch(patches.Rectangle((x, y), w*W, h*H, fill=False, linewidth=1.5, edgecolor='lime'))
        ax.text(x, y-3, CLASSES[int(cid)], color='lime', fontsize=7)

train_img = DST / 'images' / 'train'; train_lbl = DST / 'labels' / 'train'
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(12, 4 * len(CLASSES)))
for row, c in enumerate(CLASSES):
    picks = random.sample(list(train_img.glob(f'{c}_*.jpg')), 3)
    for ax, p in zip(axes[row], picks):
        draw(ax, p, train_lbl / (p.stem + '.txt'))
plt.tight_layout(); plt.show()

**Cell 12**

## 2.5 Emit `data.yaml` for Ultralytics

In [ ]:
# Cell 13 — write data.yaml
cfg = {
    'path': str(DST.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'names': {i: c for i, c in enumerate(CLASSES)},
}
(DST / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print((DST / 'data.yaml').read_text())

**Cell 14**

### Acceptance criteria
- Cell 7 shows every class populated in `train` and at least one of `val` / `test`
- Cell 9 returns an empty issues table
- Cell 11 shows correct labels overlaid for all four classes
- `data/dataset/data.yaml` exists

Proceed to **notebook 03** for dataset-health diagnostics.